# Dates Analysis

In [4]:
import pandas as pd

def analyze_Dates():
    print("\n")
    print("ANALISI LATENZE E COMPORTAMENTI")
    print("\n")

    founders_data = pd.read_csv("/home/andri/conspiracy-founders/data/interim/conspiracy_founders_clean.csv", index_col=0)
    dates_data = pd.read_csv("/home/andri/conspiracy-founders/data/interim/all_dates.csv", index_col=0)
    
    df = pd.merge(founders_data, dates_data, how="inner", on="subreddit_name")

    df.rename(columns={
        "post_creation_date": "founder_first_post",
        "ourFounder_earliest_comment": "founder_first_comment",
        "subreddit_earliest_post": "sub_first_post",
        "subreddit_earliest_comment": "sub_first_comment",
        "subreddit_creation_date_y": "sub_creation_date",
        "subreddit_earliest_post_author": "sub_first_post_author",
        "subreddit_earliest_comment_author": "sub_first_comment_author"
    }, inplace=True, errors='ignore')

    date_cols = [
        "sub_creation_date", "sub_first_post", "sub_first_comment", 
        "founder_first_post", "founder_first_comment"
    ]
    
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], unit='s', errors='coerce')
            df.loc[df[col].dt.year == 1970, col] = pd.NaT

    cols_sub = ["sub_first_comment", "sub_first_post", "sub_creation_date"]
    df["sub_moment_zero"] = df[cols_sub].min(axis=1)

    cols_founder = ["founder_first_post", "founder_first_comment"]
    df["founder_moment_zero"] = df[cols_founder].min(axis=1)

    df["latency_founder_any_activity"] = (df["founder_moment_zero"] - df["sub_moment_zero"]).dt.total_seconds() / 86400
    
    df["latency_founder_first_post"] = (df["founder_first_post"] - df["sub_moment_zero"]).dt.total_seconds() / 86400

    print("DISTRIBUZIONE LATENZA A: Qualsiasi Attività vs Subreddit")
    above_7   = len(df[df["latency_founder_any_activity"] > 7])
    above_30  = len(df[df["latency_founder_any_activity"] >= 30])
    above_150 = len(df[df["latency_founder_any_activity"] >= 150])
    above_365 = len(df[df["latency_founder_any_activity"] >= 365])
    
    print(f"> 7 Giorni: {above_7} subreddit")
    print(f"> 30 Giorni: {above_30} subreddit")
    print(f"> 150 Giorni: {above_150} subreddit")
    print(f"> 365 Giorni: {above_365} subreddit\n")

    print("DISTRIBUZIONE LATENZA B: Solo Post Fondatore vs Subreddit")
    post_above_7 = len(df[df["latency_founder_first_post"] > 7])
    post_above_30 = len(df[df["latency_founder_first_post"] >= 30])
    post_above_150 = len(df[df["latency_founder_first_post"] >= 150])
    post_above_365 = len(df[df["latency_founder_first_post"] >= 365])
    
    print(f"> 7 Giorni (Solo post): {post_above_7} subreddit")
    print(f"> 30 Giorni (Solo post): {post_above_30} subreddit")
    print(f"> 150 Giorni (Solo post): {post_above_150} subreddit")
    print(f"> 365 Giorni (Solo post): {post_above_365} subreddit\n")


    print("CONTROLLI COMPORTAMENTALI")
    
    cond_founder_comment_first = (df["founder_first_comment"] < df["founder_first_post"]) & df["founder_first_comment"].notna()
    df_founder_comm_first = df[cond_founder_comment_first]
    print(f"1. Fondatori che hanno commentato prima di fare un post: {len(df_founder_comm_first)}")

    cond_sub_comment_first = (df["sub_first_comment"] < df["sub_first_post"]) & df["sub_first_comment"].notna()
    df_sub_comm_first = df[cond_sub_comment_first]
    print(f"2. Subreddit con un commento che precede il primo post: {len(df_sub_comm_first)}")

    if "sub_first_comment_author" in df.columns and "sub_first_post_author" in df.columns:
        df_diff_authors = df_sub_comm_first[
            df_sub_comm_first["sub_first_comment_author"] != df_sub_comm_first["sub_first_post_author"]
        ]
        print(f"3. Tra questi (punto 2), l'autore del commento è diverso da quello del post in {len(df_diff_authors)} casi.")
    else:
        print("3. [Saltato] Le colonne degli autori (sub_first_comment_author, ecc.) non sono state trovate nel merge.")

    print("\n\n")

analyze_Dates()



ANALISI LATENZE E COMPORTAMENTI


DISTRIBUZIONE LATENZA A: Qualsiasi Attività vs Subreddit
> 7 Giorni: 781 subreddit
> 30 Giorni: 598 subreddit
> 150 Giorni: 339 subreddit
> 365 Giorni: 160 subreddit

DISTRIBUZIONE LATENZA B: Solo Post Fondatore vs Subreddit
> 7 Giorni (Solo post): 781 subreddit
> 30 Giorni (Solo post): 598 subreddit
> 150 Giorni (Solo post): 339 subreddit
> 365 Giorni (Solo post): 160 subreddit

CONTROLLI COMPORTAMENTALI
1. Fondatori che hanno commentato prima di fare un post: 1
2. Subreddit con un commento che precede il primo post: 5
3. Tra questi (punto 2), l'autore del commento è diverso da quello del post in 4 casi.



